# Ablation Study Analysis

This notebook analyzes ablation study results to evaluate the contribution of different components in our multimodal stock prediction framework.

## Ablation Configurations:
- **Full Model (Tech + Sent 70:30)**: Technical indicators + Sentiment (70% news, 30% social) - baseline from v1/
- **Technical Only**: Technical indicators only (no sentiment features)
- **Sentiment Only**: Sentiment features only (no technical indicators)
- **Equal Weights (50:50)**: Equal weighting of news and social media sentiment
- **News Only**: News sentiment only (no social media)
- **Social Media Only**: Social media sentiment only (no news)

In [1]:
import pandas as pd
import numpy as np
import os
import glob

# Configuration
STOCKS = ["AAPL", "META", "NVDA", "SPY", "TSLA"]
ABLATION_DIR = "../results/ablation"
V1_DIR = "v1"

# Load ablation results from individual files
def load_ablation_results():
    """Load and combine all ablation result files."""
    ablation_files = glob.glob(os.path.join(ABLATION_DIR, "ablation_*_all_stocks_results.csv"))
    
    if not ablation_files:
        print(f"No ablation result files found in {ABLATION_DIR}")
        return None
    
    all_results = []
    for f in ablation_files:
        df = pd.read_csv(f)
        all_results.append(df)
        config = df['AblationConfig'].iloc[0] if 'AblationConfig' in df.columns else 'unknown'
        print(f"Loaded {len(df)} rows from {os.path.basename(f)} (config: {config})")
    
    combined = pd.concat(all_results, ignore_index=True)
    return combined

ablation_df = load_ablation_results()

if ablation_df is not None:
    print(f"\nTotal ablation results: {len(ablation_df)}")
    print(f"Configurations: {ablation_df['AblationConfig'].unique()}")
    print(f"Stocks: {ablation_df['Stock'].unique()}")
    print(f"Models: {ablation_df['Model'].unique()}")
else:
    print("No ablation results found")

Loaded 21 rows from ablation_technical_only_all_stocks_results.csv (config: technical_only)
Loaded 21 rows from ablation_sentiment_only_all_stocks_results.csv (config: sentiment_only)
Loaded 21 rows from ablation_social_only_all_stocks_results.csv (config: social_only)
Loaded 21 rows from ablation_news_only_all_stocks_results.csv (config: news_only)
Loaded 21 rows from ablation_equal_weights_all_stocks_results.csv (config: equal_weights)

Total ablation results: 105
Configurations: ['technical_only' 'sentiment_only' 'social_only' 'news_only'
 'equal_weights']
Stocks: ['AAPL' 'META' 'NVDA' 'SPY' 'TSLA']
Models: ['LSTM' 'XGBoost' 'LogisticRegression' 'SVM' 'LightGBM' 'GradientBoosting'
 'RandomForest']


In [2]:
# Load Full Model (baseline) results from v1/ for comparison
def load_full_model_results():
    """Load and combine all v1/ results as the Full Model baseline (long_short strategy only)."""
    import glob
    all_results = []
    
    # Get all model result files (excluding strategy variants and all_stocks)
    models = ['GradientBoosting', 'RandomForest', 'XGBoost', 'LightGBM', 
              'LogisticRegression', 'SVM', 'LSTM']
    
    for model in models:
        for stock in STOCKS:
            # Only load the default long_short results (no suffix)
            file_path = os.path.join(V1_DIR, f"{model}_tuned_{stock}_results.csv")
            if os.path.exists(file_path):
                df = pd.read_csv(file_path)
                df['Model'] = model
                df['AblationConfig'] = 'full_model'
                all_results.append(df)
    
    if all_results:
        combined = pd.concat(all_results, ignore_index=True)
        # Standardize column names to match ablation format
        rename_map = {
            'Test_Accuracy': 'Test_Accuracy',
            'Test_ROC_AUC': 'Test_ROC_AUC',
            'Trades': 'Trades',
            'WinRate': 'WinRate',
            'Sharpe': 'Sharpe', 
            'TotalReturn': 'TotalReturn'
        }
        return combined
    return None

full_model_df = load_full_model_results()
if full_model_df is not None:
    print(f"Loaded {len(full_model_df)} Full Model (v1/) results")
    print(f"Models: {full_model_df['Model'].unique()}")
    print(f"Stocks: {full_model_df['Stock'].unique()}")
else:
    print("No full model results found")

No full model results found


## Ablation Analysis: Both Selection Criteria (AUC and Sharpe)

For each ablation configuration, we show results under both selection criteria:
- **AUC**: Best model selected by ROC-AUC (discriminative ability)
- **Sharpe**: Best model selected by Sharpe ratio (trading performance)

In [3]:
def compute_best_by_criterion(df, criterion='Sharpe', configs=None):
    """
    For each configuration & stock, select the best model by criterion,
    then average across stocks.
    """
    if df is None or len(df) == 0:
        return None
    
    if configs is None:
        configs = df['AblationConfig'].unique()
    
    results = []
    for config in configs:
        config_df = df[df['AblationConfig'] == config]
        if len(config_df) == 0:
            continue
        
        # Select best model per stock
        best_per_stock = []
        for stock in STOCKS:
            stock_data = config_df[config_df['Stock'] == stock]
            if len(stock_data) > 0:
                best_idx = stock_data[criterion].idxmax()
                best_per_stock.append(stock_data.loc[best_idx])
        
        if len(best_per_stock) > 0:
            best_df = pd.DataFrame(best_per_stock)
            results.append({
                'Configuration': config,
                'Accuracy': best_df['Test_Accuracy'].mean(),
                'AUC': best_df['Test_ROC_AUC'].mean(),
                'N': best_df['Trades'].mean(),
                'Win%': best_df['WinRate'].mean() * 100,
                'Sharpe': best_df['Sharpe'].mean(),
                'Ret%': best_df['TotalReturn'].mean() * 100,
                'n_stocks': len(best_per_stock)
            })
    
    return pd.DataFrame(results)

# Compute ablation results for BOTH selection criteria
ablation_by_auc = None
ablation_by_sharpe = None

if ablation_df is not None:
    ablation_by_auc = compute_best_by_criterion(ablation_df, 'Test_ROC_AUC')
    ablation_by_sharpe = compute_best_by_criterion(ablation_df, 'Sharpe')
    
    print("Ablation Results (Best by AUC):")
    if ablation_by_auc is not None:
        print(ablation_by_auc.to_string(index=False))
    print()
    
    print("Ablation Results (Best by Sharpe):")
    if ablation_by_sharpe is not None:
        print(ablation_by_sharpe.to_string(index=False))
else:
    print("No ablation results to analyze")

Ablation Results (Best by AUC):
 Configuration  Accuracy      AUC    N      Win%    Sharpe       Ret%  n_stocks
technical_only  0.484121 0.691869 22.2 37.667725 -1.133872 -36.811703         5
sentiment_only  0.506423 0.618820 32.0 52.687607  0.295736  16.663302         5
   social_only  0.488392 0.663113 24.8 42.533333 -0.811399  -4.694459         5
     news_only  0.481275 0.667543 30.4 38.651852 -0.699824   3.332625         5
 equal_weights  0.494687 0.678408 26.4 38.367981 -0.539873  23.461138         5

Ablation Results (Best by Sharpe):
 Configuration  Accuracy      AUC    N      Win%    Sharpe       Ret%  n_stocks
technical_only  0.476571 0.678074 24.6 41.039721 -0.695044 -16.123697         5
sentiment_only  0.531395 0.582513 33.6 57.604779  1.233141  39.454554         5
   social_only  0.490891 0.615984 36.4 47.189226  0.079299  49.788053         5
     news_only  0.469323 0.660011 33.2 43.672162 -0.383661  11.039580         5
 equal_weights  0.474197 0.645517 26.2 46.133333 -0.

In [4]:
# Compute Full Model baseline for BOTH selection criteria
full_model_by_auc = None
full_model_by_sharpe = None

if full_model_df is not None:
    full_model_by_auc = compute_best_by_criterion(full_model_df, 'Test_ROC_AUC')
    full_model_by_sharpe = compute_best_by_criterion(full_model_df, 'Sharpe')
    
    print("Full Model (v1/) Results (Best by AUC):")
    if full_model_by_auc is not None:
        print(full_model_by_auc.to_string(index=False))
    print()
    
    print("Full Model (v1/) Results (Best by Sharpe):")
    if full_model_by_sharpe is not None:
        print(full_model_by_sharpe.to_string(index=False))
else:
    print("No full model data available")

No full model data available


## Combined Ablation Table (for main.tex)

Table format similar to Table V, showing both AUC and Sharpe selection criteria for each configuration.

In [5]:
# Create combined ablation table with both selection criteria (like Table V)
def create_combined_table(ablation_by_auc, ablation_by_sharpe, 
                          full_model_by_auc, full_model_by_sharpe):
    """Create table with both AUC and Sharpe selection for each config."""
    
    # Configuration display names (shorter)
    config_names = {
        'full_model': 'Full Model',
        'technical_only': 'Technical Only',
        'sentiment_only': 'Sentiment Only',
        'equal_weights': 'Equal Weights',
        'news_only': 'News Only',
        'social_only': 'Social Only'
    }
    
    # Get full model baselines for delta calculation
    fm_auc_baseline = full_model_by_auc['AUC'].iloc[0] if full_model_by_auc is not None and len(full_model_by_auc) > 0 else None
    fm_sharpe_baseline = full_model_by_sharpe['Sharpe'].iloc[0] if full_model_by_sharpe is not None and len(full_model_by_sharpe) > 0 else None
    
    rows = []
    
    # Add full model rows (both criteria)
    if full_model_by_auc is not None and len(full_model_by_auc) > 0:
        fm = full_model_by_auc.iloc[0]
        rows.append({
            'Configuration': config_names.get('full_model', 'Full Model'),
            'Sel.': 'AUC',
            'Acc.': fm['Accuracy'],
            'AUC': fm['AUC'],
            'N': fm['N'],
            'Win%': fm['Win%'],
            'Sharpe': fm['Sharpe'],
            'Ret%': fm['Ret%'],
            'Delta': None  # Baseline, no delta
        })
    
    if full_model_by_sharpe is not None and len(full_model_by_sharpe) > 0:
        fm = full_model_by_sharpe.iloc[0]
        rows.append({
            'Configuration': '',  # Same config, different selection
            'Sel.': 'Sharpe',
            'Acc.': fm['Accuracy'],
            'AUC': fm['AUC'],
            'N': fm['N'],
            'Win%': fm['Win%'],
            'Sharpe': fm['Sharpe'],
            'Ret%': fm['Ret%'],
            'Delta': None  # Baseline, no delta
        })
    
    # Add ablation configurations (both criteria for each)
    configs_order = ['technical_only', 'sentiment_only', 'equal_weights', 'news_only', 'social_only']
    
    for config in configs_order:
        # AUC selection - delta is relative AUC change (%)
        if ablation_by_auc is not None:
            config_row = ablation_by_auc[ablation_by_auc['Configuration'] == config]
            if len(config_row) > 0:
                r = config_row.iloc[0]
                if fm_auc_baseline and fm_auc_baseline != 0:
                    delta_auc = ((r['AUC'] - fm_auc_baseline) / fm_auc_baseline * 100)
                else:
                    delta_auc = None
                rows.append({
                    'Configuration': config_names.get(config, config),
                    'Sel.': 'AUC',
                    'Acc.': r['Accuracy'],
                    'AUC': r['AUC'],
                    'N': r['N'],
                    'Win%': r['Win%'],
                    'Sharpe': r['Sharpe'],
                    'Ret%': r['Ret%'],
                    'Delta': delta_auc
                })
        
        # Sharpe selection - delta is relative Sharpe change (%)
        if ablation_by_sharpe is not None:
            config_row = ablation_by_sharpe[ablation_by_sharpe['Configuration'] == config]
            if len(config_row) > 0:
                r = config_row.iloc[0]
                if fm_sharpe_baseline and fm_sharpe_baseline != 0:
                    delta_sharpe = ((r['Sharpe'] - fm_sharpe_baseline) / abs(fm_sharpe_baseline) * 100)
                else:
                    delta_sharpe = None
                rows.append({
                    'Configuration': '',  # Same config
                    'Sel.': 'Sharpe',
                    'Acc.': r['Accuracy'],
                    'AUC': r['AUC'],
                    'N': r['N'],
                    'Win%': r['Win%'],
                    'Sharpe': r['Sharpe'],
                    'Ret%': r['Ret%'],
                    'Delta': delta_sharpe
                })
    
    return pd.DataFrame(rows)

# Create combined table
combined_table = create_combined_table(ablation_by_auc, ablation_by_sharpe,
                                        full_model_by_auc, full_model_by_sharpe)

print("Combined Ablation Table (Both Selection Criteria):")
print(combined_table.to_string(index=False))
print()

# Find max values for each column (for bolding)
max_acc = combined_table['Acc.'].max()
max_auc = combined_table['AUC'].max()
max_win = combined_table['Win%'].max()
max_sharpe = combined_table['Sharpe'].max()
max_ret = combined_table['Ret%'].max()

def format_val(val, max_val, fmt):
    """Format value, bold if it's the max."""
    if pd.isna(val):
        return "--"
    formatted = fmt.format(val)
    if abs(val - max_val) < 0.001:  # Close enough to max
        return f"\\textbf{{{formatted}}}"
    return formatted

def format_delta(delta):
    """Format delta value as number (no % sign)."""
    if delta is None or pd.isna(delta):
        return "--"
    return f"{delta:+.1f}"

# Generate compact LaTeX table with bold max values
print("\nLaTeX Table:")
print("=" * 70)
print(r"""\begin{table}[htbp]
\caption{Ablation Study Results}
\label{tab:ablation}
\centering
\resizebox{\linewidth}{!}{%
\begin{tabular}{llcccccccc}
\toprule
Configuration & Sel. & Acc. & AUC & $N$ & Win\% & Sharpe & Ret\% & $\Delta$(\%) \\
\midrule""")

prev_config = None
for _, row in combined_table.iterrows():
    config = row['Configuration'] if row['Configuration'] else ''
    
    # Add midrule between different configs
    if prev_config is not None and row['Configuration'] != '' and row['Configuration'] != prev_config:
        print(r"\midrule")
    
    # Format values with bold for max
    acc_str = format_val(row['Acc.'], max_acc, "{:.3f}")
    auc_str = format_val(row['AUC'], max_auc, "{:.3f}")
    n_str = f"{row['N']:.1f}"
    win_str = format_val(row['Win%'], max_win, "{:.1f}")
    sharpe_str = format_val(row['Sharpe'], max_sharpe, "{:.2f}")
    ret_str = format_val(row['Ret%'], max_ret, "{:.1f}")
    delta_str = format_delta(row['Delta'])
    
    # Format the row
    if config:
        print(f"{config} & {row['Sel.']} & {acc_str} & {auc_str} & {n_str} & {win_str} & {sharpe_str} & {ret_str} & {delta_str} \\\\")
    else:
        print(f"\\quad & {row['Sel.']} & {acc_str} & {auc_str} & {n_str} & {win_str} & {sharpe_str} & {ret_str} & {delta_str} \\\\")
    
    if row['Configuration']:
        prev_config = row['Configuration']

print(r"""\bottomrule
\multicolumn{9}{l}{\footnotesize $\Delta$ = $\Delta$AUC when Sel.=AUC; $\Delta$Sharpe when Sel.=Sharpe.}
\end{tabular}
}
\end{table}""")

Combined Ablation Table (Both Selection Criteria):
 Configuration   Sel.     Acc.      AUC    N      Win%    Sharpe       Ret% Delta
Technical Only    AUC 0.484121 0.691869 22.2 37.667725 -1.133872 -36.811703  None
               Sharpe 0.476571 0.678074 24.6 41.039721 -0.695044 -16.123697  None
Sentiment Only    AUC 0.506423 0.618820 32.0 52.687607  0.295736  16.663302  None
               Sharpe 0.531395 0.582513 33.6 57.604779  1.233141  39.454554  None
 Equal Weights    AUC 0.494687 0.678408 26.4 38.367981 -0.539873  23.461138  None
               Sharpe 0.474197 0.645517 26.2 46.133333 -0.248785  28.629622  None
     News Only    AUC 0.481275 0.667543 30.4 38.651852 -0.699824   3.332625  None
               Sharpe 0.469323 0.660011 33.2 43.672162 -0.383661  11.039580  None
   Social Only    AUC 0.488392 0.663113 24.8 42.533333 -0.811399  -4.694459  None
               Sharpe 0.490891 0.615984 36.4 47.189226  0.079299  49.788053  None


LaTeX Table:
\begin{table}[htbp]
\caption{Abl

## Per-Stock Breakdown

Detailed results for each stock showing both selection criteria.

In [6]:
# Per-stock best for each config and criterion
def get_per_stock_best(df, criterion='Sharpe'):
    """Get best model per stock for each configuration."""
    if df is None:
        return None
    
    results = []
    for config in df['AblationConfig'].unique():
        config_df = df[df['AblationConfig'] == config]
        for stock in STOCKS:
            stock_data = config_df[config_df['Stock'] == stock]
            if len(stock_data) > 0:
                best_idx = stock_data[criterion].idxmax()
                best = stock_data.loc[best_idx]
                results.append({
                    'Config': config,
                    'Stock': stock,
                    'Model': best['Model'],
                    'Horizon': best['Horizon'],
                    'AUC': best['Test_ROC_AUC'],
                    'Sharpe': best['Sharpe'],
                    'Ret%': best['TotalReturn'] * 100,
                    'Win%': best['WinRate'] * 100
                })
    
    return pd.DataFrame(results)

# Combine full model and ablation results
all_results_df = None
if full_model_df is not None and ablation_df is not None:
    all_results_df = pd.concat([full_model_df, ablation_df], ignore_index=True)
elif full_model_df is not None:
    all_results_df = full_model_df
elif ablation_df is not None:
    all_results_df = ablation_df

if all_results_df is not None:
    config_order = ['full_model', 'technical_only', 'sentiment_only', 
                    'equal_weights', 'news_only', 'social_only']
    
    # Best by AUC
    per_stock_auc = get_per_stock_best(all_results_df, 'Test_ROC_AUC')
    if per_stock_auc is not None:
        print("="*60)
        print("BEST BY AUC")
        print("="*60)
        
        print("\nPer-Stock AUC:")
        pivot_auc = per_stock_auc.pivot(index='Config', columns='Stock', values='AUC')
        pivot_auc = pivot_auc.reindex([c for c in config_order if c in pivot_auc.index])
        pivot_auc['AVG'] = pivot_auc.mean(axis=1)
        print(pivot_auc.round(3).to_string())
        
        print("\nPer-Stock Sharpe (when selected by AUC):")
        pivot_sharpe = per_stock_auc.pivot(index='Config', columns='Stock', values='Sharpe')
        pivot_sharpe = pivot_sharpe.reindex([c for c in config_order if c in pivot_sharpe.index])
        pivot_sharpe['AVG'] = pivot_sharpe.mean(axis=1)
        print(pivot_sharpe.round(2).to_string())
    
    # Best by Sharpe
    per_stock_sharpe = get_per_stock_best(all_results_df, 'Sharpe')
    if per_stock_sharpe is not None:
        print("\n" + "="*60)
        print("BEST BY SHARPE")
        print("="*60)
        
        print("\nPer-Stock Sharpe:")
        pivot_sharpe = per_stock_sharpe.pivot(index='Config', columns='Stock', values='Sharpe')
        pivot_sharpe = pivot_sharpe.reindex([c for c in config_order if c in pivot_sharpe.index])
        pivot_sharpe['AVG'] = pivot_sharpe.mean(axis=1)
        print(pivot_sharpe.round(2).to_string())
        
        print("\nPer-Stock Return%:")
        pivot_ret = per_stock_sharpe.pivot(index='Config', columns='Stock', values='Ret%')
        pivot_ret = pivot_ret.reindex([c for c in config_order if c in pivot_ret.index])
        pivot_ret['AVG'] = pivot_ret.mean(axis=1)
        print(pivot_ret.round(1).to_string())
        
        print("\nModels Selected (by Sharpe):")
        pivot_model = per_stock_sharpe.pivot(index='Config', columns='Stock', values='Model')
        pivot_model = pivot_model.reindex([c for c in config_order if c in pivot_model.index])
        print(pivot_model.to_string())

BEST BY AUC

Per-Stock AUC:
Stock            AAPL   META   NVDA    SPY   TSLA    AVG
Config                                                  
technical_only  0.550  0.771  0.887  0.717  0.534  0.692
sentiment_only  0.594  0.572  0.717  0.582  0.629  0.619
equal_weights   0.650  0.740  0.761  0.715  0.525  0.678
news_only       0.567  0.773  0.765  0.712  0.521  0.668
social_only     0.549  0.772  0.762  0.717  0.515  0.663

Per-Stock Sharpe (when selected by AUC):
Stock           AAPL  META  NVDA   SPY  TSLA   AVG
Config                                            
technical_only -1.45 -1.44 -0.24 -1.86 -0.69 -1.13
sentiment_only  1.45  0.96  1.45 -1.86 -0.52  0.30
equal_weights   0.90 -1.56 -2.12 -2.05  2.14 -0.54
news_only      -0.23 -1.44 -1.76 -1.86  1.79 -0.70
social_only    -0.67 -1.44 -1.76 -1.86  1.67 -0.81

BEST BY SHARPE

Per-Stock Sharpe:
Stock           AAPL  META  NVDA   SPY  TSLA   AVG
Config                                            
technical_only -0.85 -1.44 -0.24 -1.8

## Raw Data Preview

Show the raw ablation results data.

In [7]:
# Display raw ablation results
if ablation_df is not None:
    display_cols = ['Stock', 'Horizon', 'AblationConfig', 'Model', 'Test_Accuracy', 
                    'Test_ROC_AUC', 'Trades', 'WinRate', 'Sharpe', 'TotalReturn']
    available_cols = [c for c in display_cols if c in ablation_df.columns]
    print(f"Total rows: {len(ablation_df)}")
    print(ablation_df[available_cols].to_string(index=False))

Total rows: 105
Stock  Horizon AblationConfig              Model  Test_Accuracy  Test_ROC_AUC  Trades  WinRate    Sharpe  TotalReturn
 AAPL        4 technical_only               LSTM       0.447964      0.508114      55 0.363636 -1.609474    -0.311411
 AAPL        5 technical_only            XGBoost       0.482072      0.534711      50 0.400000 -1.334173    -0.303248
 AAPL        6 technical_only            XGBoost       0.517928      0.506129      41 0.463415 -0.847177    -0.175790
 AAPL        8 technical_only               LSTM       0.515837      0.550295      27 0.370370 -1.448200    -0.300365
 AAPL       10 technical_only               LSTM       0.488688      0.545149      22 0.409091 -1.519379    -0.288297
 META        2 technical_only LogisticRegression       0.450199      0.622226     125 0.408000 -1.778811    -0.485120
 META        3 technical_only LogisticRegression       0.454183      0.636221      83 0.349398 -1.792929    -0.470699
 META        4 technical_only LogisticRe

## Summary Statistics

Overall statistics for each ablation configuration.

In [8]:
# Summary statistics by ablation config (including full model)
# Combine full model and ablation results for summary
all_df = None
if full_model_df is not None and ablation_df is not None:
    all_df = pd.concat([full_model_df, ablation_df], ignore_index=True)
elif full_model_df is not None:
    all_df = full_model_df
elif ablation_df is not None:
    all_df = ablation_df

if all_df is not None:
    summary = all_df.groupby('AblationConfig').agg({
        'Test_Accuracy': ['mean', 'std', 'count'],
        'Test_ROC_AUC': ['mean', 'std', 'max'],
        'Sharpe': ['mean', 'std'],
        'TotalReturn': ['mean', 'std']
    }).round(3)
    
    # Reorder to put full_model first
    config_order = ['full_model', 'technical_only', 'equal_weights', 'sentiment_only', 'news_only', 'social_only']
    available_configs = [c for c in config_order if c in summary.index]
    summary = summary.reindex(available_configs)
    
    print("Summary Statistics by Configuration (including Full Model):")
    print(summary.to_string())
    
    print("\n\nStocks coverage per config:")
    for config in available_configs:
        config_stocks = all_df[all_df['AblationConfig'] == config]['Stock'].unique()
        missing = [s for s in STOCKS if s not in config_stocks]
        if missing:
            print(f"  {config}: Missing {missing}")
        else:
            print(f"  {config}: All 5 stocks present")

Summary Statistics by Configuration (including Full Model):
               Test_Accuracy              Test_ROC_AUC               Sharpe        TotalReturn       
                        mean    std count         mean    std    max   mean    std        mean    std
AblationConfig                                                                                       
technical_only         0.472  0.045    21        0.649  0.112  0.887 -1.427  0.814      -0.343  0.222
equal_weights          0.482  0.033    21        0.638  0.090  0.761 -0.964  1.557      -0.047  0.843
sentiment_only         0.512  0.039    21        0.562  0.052  0.717  0.079  1.155       0.046  0.419
news_only              0.479  0.029    21        0.638  0.099  0.773 -1.206  1.283      -0.183  0.562
social_only            0.485  0.032    21        0.627  0.099  0.772 -0.884  1.444      -0.052  0.817


Stocks coverage per config:
  technical_only: All 5 stocks present
  equal_weights: All 5 stocks present
  sentiment_only:

In [14]:
!cd .. && python scripts/update_sota_table.py --update-ablation

UPDATING SOTA TABLE
Results directory: results/baselines
LaTeX file: docs/main.tex
Ours results file: results/ours/tuned_all_results_combined.csv

Loading Chronos-2 results...
Loaded 180 Chronos-2 result rows
Model types: ['zero-shot' 'multivariate' 'finetuned' 'finetuned_cov']
Stocks: ['AAPL' 'META' 'NVDA' 'SPY' 'TSLA']

Loading FinCast results...
  Loaded 45 zero-shot results
  Loaded 45 fine-tuned results
Loaded 90 FinCast result rows
Model types: ['zero-shot' 'finetuned']
Stocks: ['AAPL' 'META' 'NVDA' 'SPY' 'TSLA']

Computing 'Ours' metrics from tuned results...
Ours metrics: Acc=0.523, AUC=0.565, N=48.0, Win%=66.0, Sharpe=2.06

Calculating baseline metrics...

Baseline Metrics Summary:
--------------------------------------------------------------------------------

FinCast Baselines:

fincast_zeroshot:
  AUC-selected: Acc=0.524, AUC=0.560, Sharpe=-0.77
  Sharpe-selected: Acc=0.474, AUC=0.507, Sharpe=-0.10

fincast_finetuned:
  AUC-selected: Acc=0.596, AUC=0.693, Sharpe=-0.40
  Sh